In [10]:
# Audio inspection script for the Lanzhou 2015 dataset.

"""Loop over all subject folders, inspect every .wav file, and write a summary CSV.

Expected layout:
    <base_dir>/
        02010001/
            01.wav, 02.wav, ... 29.wav
        02010002/
            ...

For each .wav file we record: subject id, type, PHQ-9, file name, duration (s),
sample rate (Hz), and number of channels.
"""

import csv
import wave
from pathlib import Path

import pandas as pd

# Directory that contains the subject-id folders. In a notebook there is no
# __file__, so we use the current working directory (where the notebook runs).
BASE_DIR = Path.cwd()
OUTPUT_CSV = BASE_DIR / "wav_inventory.csv"
SUBJECTS_XLSX = BASE_DIR / "subjects_information_audio_lanzhou_2015.xlsx"


def load_subject_metadata() -> dict:
    """Map subject id (as int) -> metadata dict (type, PHQ-9) from the metadata sheet.

    Folder names are zero-padded (e.g. '02010001') while the spreadsheet stores
    the id as an integer (2010001), so we key the map on the int value.
    """
    df = pd.read_excel(SUBJECTS_XLSX, usecols=["subject id", "type", "PHQ-9"])
    df = df.dropna(subset=["subject id"])
    meta = {}
    for _, row in df.iterrows():
        phq9 = row["PHQ-9"]
        meta[int(row["subject id"])] = {
            "type": str(row["type"]),
            "phq9": None if pd.isna(phq9) else int(phq9),
        }
    return meta


def inspect_wav(path: Path) -> dict:
    """Return metadata for a single .wav file using the stdlib `wave` module."""
    with wave.open(str(path), "rb") as wf:
        n_frames = wf.getnframes()
        sample_rate = wf.getframerate()
        n_channels = wf.getnchannels()
        duration = n_frames / sample_rate if sample_rate else 0.0
    return {
        "sample_rate": sample_rate,
        "n_channels": n_channels,
        "duration_sec": round(duration, 3),
    }


def main() -> None:
    rows = []
    subject_meta = load_subject_metadata()
    missing_types = set()

    # Subject folders: directories whose name is all digits (the subject id).
    subject_dirs = sorted(
        d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.isdigit()
    )

    for subject_dir in subject_dirs:
        subject_id = subject_dir.name
        meta_row = subject_meta.get(int(subject_id))
        if meta_row is None:
            missing_types.add(subject_id)
        subject_type = meta_row["type"] if meta_row else None
        subject_phq9 = meta_row["phq9"] if meta_row else None
        for wav_path in sorted(subject_dir.glob("*.wav")):
            try:
                meta = inspect_wav(wav_path)
            except (wave.Error, EOFError) as exc:
                print(f"  ! Could not read {wav_path}: {exc}")
                meta = {"sample_rate": None, "n_channels": None, "duration_sec": None}
            rows.append(
                {
                    "subject_id": subject_id,
                    "type": subject_type,
                    "phq9": subject_phq9,
                    "file_name": wav_path.name,
                    "duration_sec": meta["duration_sec"],
                    "sample_rate": meta["sample_rate"],
                    "n_channels": meta["n_channels"],
                }
            )

    fieldnames = ["subject_id", "type", "phq9", "file_name", "duration_sec", "sample_rate", "n_channels"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Inspected {len(subject_dirs)} subjects, {len(rows)} .wav files.")
    if missing_types:
        print(f"  ! No 'type' found for {len(missing_types)} subjects: {sorted(missing_types)}")
    print(f"Wrote inventory to {OUTPUT_CSV}")


main()

  ! Could not read c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\02010004\24.wav: file does not start with RIFF id
  ! Could not read c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\02010004\25.wav: file does not start with RIFF id
  ! Could not read c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\02010004\26.wav: file does not start with RIFF id
  ! Could not read c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\02010004\27.wav: file does not start with RIFF id
  ! Could not read c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\02010004\28.wav: file does not start with RIFF id


KeyboardInterrupt: 

In [ ]:
# Computing summaries of the inventory (min/max/mean duration, counts by sample rate and channel count).

import pandas as pd

# Load the inventory produced by inspect_wav_files.py
df = pd.read_csv("wav_inventory.csv")

# Files that could not be read (corrupt headers) have NaN audio metadata.
readable = df.dropna(subset=["duration_sec"])
n_unreadable = len(df) - len(readable)

# --- Overall min / max / mean duration (seconds) ---
print("Duration (seconds):")
print(f"  min  = {readable['duration_sec'].min():.3f}")
print(f"  max  = {readable['duration_sec'].max():.3f}")
print(f"  mean = {readable['duration_sec'].mean():.3f}")

# --- Count of files by sample rate ---
print("\nFiles by sample rate (Hz):")
print(readable["sample_rate"].astype(int).value_counts().to_string())

# --- Count of mono vs stereo ---
channel_labels = {1: "mono", 2: "stereo"}
channel_counts = readable["n_channels"].astype(int).map(
    lambda c: channel_labels.get(c, f"{c}-channel")
).value_counts()
print("\nMono vs stereo:")
print(channel_counts.to_string())

if n_unreadable:
    print(f"\nNote: {n_unreadable} unreadable file(s) excluded from the stats above.")


Duration (seconds):
  min  = 1.357
  max  = 165.330
  mean = 17.215

Files by sample rate (Hz):
sample_rate
44100    1503

Mono vs stereo:
n_channels
mono    1503

Note: 5 unreadable file(s) excluded from the stats above.


In [ ]:
# Audio inventory cleanup: removing unreadable files and files with length < 3 seconds.

# Load the inventory. dtype=str on subject_id keeps the zero-padded folder name
# (e.g. "02010001"); without it pandas parses it as an int and drops the leading 0.
df = pd.read_csv("wav_inventory.csv", dtype={"subject_id": str})

# Remove unreadable files (corrupt headers)
df = df.dropna(subset=["duration_sec"])

# Remove files with length < 3 seconds
df = df[df["duration_sec"] >= 3]

# Save the cleaned inventory
df.to_csv("wav_inventory_cleaned.csv", index=False)
print(f"Cleaned inventory: {len(df)} files retained.")


Cleaned inventory: 1475 files retained.


In [ ]:
# Resampling audio files to 16 kHz.
# Driven by the cleaned inventory (cell 3), so corrupt and <3s files are already excluded.
# The subject-folder structure is mirrored in the output so identical file names
# (01.wav, 02.wav, ...) from different subjects do not collide.

import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path

base_dir = Path.cwd()
out_root = base_dir / "audio_lanzhou_2015_resampled"
target_sr = 16000

cleaned = pd.read_csv("wav_inventory_cleaned.csv", dtype={"subject_id": str})

n_done, n_skipped, n_failed = 0, 0, 0
for row in cleaned.itertuples(index=False):
    src = base_dir / row.subject_id / row.file_name
    out_dir = out_root / row.subject_id
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{src.stem}_resampled.wav"

    if out_path.exists():           # resumable: skip files already converted
        n_skipped += 1
        continue
    try:
        y, sr = librosa.load(src, sr=target_sr, mono=True)  # librosa resamples on load
        sf.write(out_path, y, target_sr)
        n_done += 1
    except Exception as exc:
        print(f"  ! Failed {src}: {exc}")
        n_failed += 1

print(f"Resampled {n_done} file(s) to {target_sr} Hz, skipped {n_skipped} existing, {n_failed} failed.")
print(f"Output written under {out_root}")


Resampled 0 file(s) to 16000 Hz, skipped 1475 existing, 0 failed.
Output written under c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\audio_lanzhou_2015_resampled


# eGeMAPS Feature Extraction

We then run this script through the terminal to extract eGeMAPS features from all the cleaned, resampled `.wav` files:

```powershell
# Paths
$smile = "C:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\tools\opensmile-3.0-win-x64\bin\SMILExtract.exe"
$cfg   = "C:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\tools\opensmile-3.0-win-x64\config\egemaps\v01a\eGeMAPSv01a.conf"
$root  = "C:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\audio_lanzhou_2015_resampled"

# Every resampled WAV under each subject folder
$wavs = Get-ChildItem -Path $root -Recurse -Filter *_resampled.wav

$done = 0; $skipped = 0; $failed = 0
foreach ($wav in $wavs) {
    $outDir = Join-Path $wav.DirectoryName "egemaps"
    New-Item -ItemType Directory -Force -Path $outDir | Out-Null

    $outCsv = Join-Path $outDir ($wav.BaseName + "_egemaps.csv")
    if (Test-Path $outCsv) { $skipped++; continue }   # resumable

    & $smile -C $cfg -I $wav.FullName -O $outCsv -l 1   # -l 1 = quieter logging
    if ($LASTEXITCODE -eq 0) { $done++ }
    else { Write-Warning "Failed: $($wav.FullName)"; $failed++ }
}

Write-Host "Extracted $done, skipped $skipped existing, $failed failed."
```


In [ ]:
# Parse the openSMILE eGeMAPS ARFF outputs into one feature table.
#
# Each openSMILE output is an ARFF file (despite the .csv extension): an @attribute
# header block followed by a single @data row of features for that WAV. We parse each
# one into a single row, derive subject_id / recording from the file path, drop the
# useless 'name' and 'class' columns, and merge in the clinical data (type, PHQ-9).

import csv as _csv
from pathlib import Path

import pandas as pd

base_dir = Path.cwd()
features_root = base_dir / "audio_lanzhou_2015_resampled"
OUTPUT_FEATURES = base_dir / "egemaps_features.csv"


def parse_arff(path: Path) -> dict:
    """Parse a single-instance openSMILE ARFF file into {attribute: value}."""
    attributes = []
    data_line = None
    in_data = False
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue
            if not in_data:
                low = stripped.lower()
                if low.startswith("@attribute"):
                    # "@attribute <name> <type>" -> take the 2nd token as the name
                    attributes.append(stripped.split()[1])
                elif low.startswith("@data"):
                    in_data = True
            elif data_line is None:
                data_line = stripped  # first (and only) data row

    if data_line is None:
        raise ValueError(f"No @data row found in {path}")

    # csv.reader handles the quoted 'unknown' name field correctly.
    values = next(_csv.reader([data_line], quotechar="'"))
    if len(values) != len(attributes):
        raise ValueError(
            f"{path}: {len(values)} values vs {len(attributes)} attributes"
        )

    row = {}
    for attr, val in zip(attributes, values):
        if attr in ("name", "class"):   # openSMILE bookkeeping columns -> drop
            continue
        row[attr] = pd.NA if val == "?" else float(val)
    return row


# Subject-level clinical data (type, PHQ-9) from the inventory we built earlier.
inv = pd.read_csv("wav_inventory.csv", dtype={"subject_id": str})
clinical = (
    inv[["subject_id", "type", "phq9"]]
    .drop_duplicates("subject_id")
    .set_index("subject_id")
)

rows = []
arff_files = sorted(features_root.glob("*/egemaps/*_egemaps.*"))
for arff in arff_files:
    subject_id = arff.parents[1].name                 # .../<subject_id>/egemaps/<file>
    recording = arff.stem.replace("_resampled_egemaps", "")  # e.g. "01"
    feats = parse_arff(arff)
    meta = clinical.loc[subject_id] if subject_id in clinical.index else None
    rows.append(
        {
            "subject_id": subject_id,
            "recording": recording,
            "type": meta["type"] if meta is not None else pd.NA,
            "phq9": meta["phq9"] if meta is not None else pd.NA,
            **feats,
        }
    )

features = pd.DataFrame(rows).sort_values(["subject_id", "recording"]).reset_index(drop=True)
features.to_csv(OUTPUT_FEATURES, index=False)

print(f"Parsed {len(features)} ARFF file(s) into {features.shape[1]} columns.")
print(f"  eGeMAPS feature columns: {features.shape[1] - 4}")  # minus id/recording/type/phq9
print(f"Wrote feature table to {OUTPUT_FEATURES}")
features.head()


Parsed 1475 ARFF file(s) into 92 columns.
  eGeMAPS feature columns: 88
Wrote feature table to c:\Users\zeine\OneDrive\Documents\bachelor thesis\data\audio_lanzhou_2015\audio_lanzhou_2015\egemaps_features.csv


,subject_id,recording,type,phq9,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,02010001,01,MDD,21,25.01068,0.064865,23.32925,25.09174,26.45478,3.125528,...,-0.027109,0.014128,0.005139,1.601602,1.307847,0.143077,0.112413,0.649167,0.498890,-55.35606
1,02010001,02,MDD,21,26.17538,0.071939,24.67093,25.74669,28.09716,3.426228,...,-0.032314,0.014758,0.004681,1.673640,0.845666,0.165000,0.160702,0.788000,0.642165,-56.49762
2,02010001,05,MDD,21,26.47824,0.045000,26.33439,26.64841,27.40425,1.069862,...,-0.036043,0.014315,0.004678,1.510574,0.613497,0.165000,0.125000,0.960000,0.617306,-57.94426
3,02010001,06,MDD,21,24.84865,0.062309,23.28705,25.12460,26.20651,2.919458,...,-0.029416,0.013902,0.005942,2.447164,1.230425,0.118182,0.085154,0.672727,0.837790,-56.52605
4,02010001,07,MDD,21,24.46347,0.124743,24.08880,25.10522,26.30504,2.216240,...,-0.035494,0.014967,0.005762,1.859800,1.008646,0.197143,0.131986,0.771429,0.643837,-56.76086


In [2]:
# Merging the audio predictors with the clinical predictors and the PHQ-9 target variable.

from pathlib import Path
import pandas as pd

# --- 0. Anchor to the data root, regardless of where the notebook sits ---
# Notebook lives in <data_root>/scripts/, so the data files are one level up.
DATA_DIR = Path.cwd()
if not (DATA_DIR / "subjects_information_audio_lanzhou_2015.xlsx").exists():
    DATA_DIR = DATA_DIR.parent

# --- 1. Clinical predictors + PHQ-9 target (per subject) ---
clinical = pd.read_excel(DATA_DIR / "subjects_information_audio_lanzhou_2015.xlsx")

clinical["subject_key"] = clinical["subject id"].astype("Int64")
clinical = clinical.rename(columns={
    "education（years）": "education_years",
    "PHQ-9": "phq9",
    "CTQ-SF": "ctq_sf",
    "GAD-7": "gad7",
})

clinical_cols = [
    "subject_key", "phq9",                              # target
    "age", "gender", "education_years",                 # demographics
    "ctq_sf", "LES", "SSRS", "gad7", "PSQI",            # other clinical scales
]
clinical = clinical[clinical_cols].dropna(subset=["subject_key"])
clinical["gender"] = clinical["gender"].map({"F": 0, "M": 1})

# --- 2. Acoustic predictors (per recording -> aggregate to per subject) ---
acoustic = pd.read_csv(DATA_DIR / "egemaps_features.csv", dtype={"subject_id": str})
acoustic["subject_key"] = acoustic["subject_id"].astype(int)

feature_cols = [c for c in acoustic.columns
                if c not in ("subject_id", "recording", "type", "phq9", "subject_key")]
acoustic_agg = acoustic.groupby("subject_key")[feature_cols].mean().reset_index()

# --- 3. Merge into one modeling table ---
model_df = clinical.merge(acoustic_agg, on="subject_key", how="inner")

print(f"Modeling table: {model_df.shape[0]} subjects, "
      f"{len(clinical_cols) - 2} clinical + {len(feature_cols)} acoustic predictors")

y = model_df["phq9"]
X_clinical = model_df[["age", "gender", "education_years", "ctq_sf", "LES", "SSRS", "gad7", "PSQI"]]
X_acoustic = model_df[feature_cols]
X = pd.concat([X_clinical, X_acoustic], axis=1)

model_df.to_csv(DATA_DIR / "model_df.csv", index=False)



Modeling table: 52 subjects, 8 clinical + 88 acoustic predictors
